# Day 30 — Solutions: Project — EDA and Preprocessing
End-to-end checklist: load → validate → clean → explore → split → save.

In [ ]:
# Imports
import pandas as pd
import seaborn as sns, matplotlib.pyplot as plt
import pandera.pandas as pa, pandera.typing as pat
from sklearn.model_selection import train_test_split
sns.set_theme(style='whitegrid')

## 1) Load data with explicit dtypes (replace paths as needed)

In [ ]:
# Example: adjust to your dataset
# dtypes = {'city':'string', 'price':'float64', 'qty':'Int64', 'date':'string'}
# df = pd.read_csv('raw.csv', dtype=dtypes, parse_dates=['date'])
# For demo, use tips
df = sns.load_dataset('tips').rename(columns=str.lower)
df.head()

## 2) Validate schema (edit to match your data)

In [ ]:
class Schema(pa.DataFrameModel):
    total_bill: pat.Series[float] = pa.Field(ge=0)
    tip: pat.Series[float] = pa.Field(ge=0)
    sex: pat.Series[str]
    smoker: pat.Series[str]
    day: pat.Series[str]
    time: pat.Series[str]
    size: pat.Series[int] = pa.Field(ge=1)

Schema.validate(df); df.dtypes

## 3) Profile nulls/dtypes/dupes/outliers

In [ ]:
summary = {
    'shape': df.shape,
    'nulls_top': df.isna().mean().sort_values(ascending=False).head(10).to_dict(),
    'dtypes': df.dtypes.astype(str).to_dict(),
    'dupes': int(df.duplicated().sum()),
}
summary

## 4) Clean and transform (example transforms)

In [ ]:
def clean(d: pd.DataFrame) -> pd.DataFrame:
    out = d.copy()
    # Standardize categorical casing
    for c in ['sex','smoker','day','time']:
        if c in out.columns:
            out[c] = out[c].astype('string').str.strip().str.lower()
    # Numeric coercions (if applicable)
    for c in ['total_bill','tip']:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors='coerce')
    # Drop rows missing key fields
    out = out.dropna(subset=['total_bill','tip','size'])
    return out

tidy = clean(df); tidy.head()

## 5) Exploratory visuals (distributions & relationships)

In [ ]:
sns.histplot(data=tidy, x='total_bill', kde=True); plt.title('Total bill distribution'); plt.tight_layout(); plt.show()
sns.scatterplot(data=tidy, x='total_bill', y='tip', hue='time'); plt.title('Tip vs total bill'); plt.tight_layout(); plt.show()

## 6) Split train/test before target-aware transforms

In [ ]:
train, test = train_test_split(tidy, test_size=0.2, random_state=42)
train.shape, test.shape

## 7) Save processed data and simple data dictionary

In [ ]:
from pathlib import Path
outdir = Path('artifacts/day30/processed')
outdir.mkdir(parents=True, exist_ok=True)
train.to_parquet(outdir/'train.parquet', index=False)
test.to_parquet(outdir/'test.parquet', index=False)
metadata = {'columns': tidy.dtypes.astype(str).to_dict(), 'rows': len(tidy)}
metadata